# Sync MongoDB => DeltaLake

Descarga OTs que no esten presentes en el DeltaLake desde MongoDB

In [ ]:
# Ubicación del Delta Talbe

DELTA_PATH = "/home/vlad/delta_v23"

### Carga librerias

In [2]:
from importlib import reload
from eerssa import gestionOT as OrdenTrabajo             # Convert from PDF_ot to obj_ot
from eerssa import matrizActividades as Actividades     # process ot.data["actividades"]

In [3]:
reload( OrdenTrabajo )
reload( Actividades  )

<module 'eerssa.matrizActividades' from '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/eerssa/matrizActividades.py'>

In [1]:
import os
import sys
import time
from pathlib import Path
import pandas as pd
import re
import json
import pickle
import logging
import pymongo
from pymongo.errors import ConnectionFailure
from deltalake import DeltaTable, write_deltalake
from pprint import pprint
from datetime import datetime

from eerssa.secret import Keys

logging.basicConfig(level=logging.INFO)

# --- MongoDB Connection ---
# It's better to establish the connection once and keep it open for the app's lifetime.
# We will also exit if the connection fails, as the consumer can't do its job without it.
uri = Keys.MONGO_KEY.value
client = None  # Initialize client to None
db_eerssa = None
CurrentCollection = None
ReloadCollection = None


try:
    # Add a timeout to avoid blocking indefinitely
    client = pymongo.MongoClient(uri, serverSelectionTimeoutMS=5000)
    # The ping command is cheap and does not require auth.
    client.admin.command('ping')
    db_eerssa = client.eerssa                   # Base de datos EERSSA
    CurrentCollection = db_eerssa.ot_v22        # Coleccion actual
    ReloadCollection  = db_eerssa.ot_reemplazo  # Aqui se cargan OTs repetidas
    logging.info(":::: Conexion exitosa con MongoDB ::::")
    
except ConnectionFailure as e:
    logging.error(f"\n\n ><><> Error de conexion a MongoDB: {e}")
    sys.exit(1) # Exit the script if we can't connect to MongoDB, as it's a critical dependency.

Success!!!


INFO:root::::: Conexion exitosa con MongoDB ::::


### Descargar ot_id desde Mongo